In [0]:
%sql

CREATE CATALOG IF NOT EXISTS rocket

In [0]:
%sql

USE CATALOG rocket;

CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, sum

# Definição de variáveis de caminho e import
catalogo = "rocket"
bronze_db_name = "bronze"

def ingest_csv(nome_arquivo, nome_tabela):
   
    try:
        table_name = nome_tabela
        landing_path = f"/Volumes/rocket/default/landing/{nome_arquivo}"

        df = spark.read.csv(landing_path, header=True, inferSchema=True, multiLine=True, escape='"')

        # Validação: arquivo vazio
        if df.count() == 0:
            raise ValueError(f"O arquivo {nome_arquivo} está vazio ou não pôde ser lido.")

        # Adiciona timestamp de ingestão
        df_with_metadata = df.withColumn("timestamp_ingestion", F.current_timestamp())

        # Escrita no formato Delta
        df_with_metadata.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{bronze_db_name}.{table_name}")

        print(f"Tabela bronze.{nome_tabela} criada com sucesso!\n")

    except Exception as e:
        print(f"Erro ao processar {nome_tabela}: {str(e)}")

In [0]:
ingest_csv('avaliacoes.csv', 'tb_avaliacoes')
ingest_csv('catalogo_produtos.csv', 'tb_catalogo_produtos')
ingest_csv('clickstream.csv', 'tb_clickstream')
ingest_csv('clientes.csv', 'tb_clientes')
ingest_csv('pedidos.csv', 'tb_pedidos')
ingest_csv('suporte_tickets.csv', 'tb_suporte_tickets')